In [32]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler , PolynomialFeatures
from sklearn.metrics import mean_absolute_error, mean_squared_error,root_mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression , Ridge,Lasso,ElasticNet
from sklearn.pipeline import Pipeline

df = pd.read_csv('../data/new_df.csv')
df.head()

,Square_Footage,Num_Bedrooms,Num_Bathrooms,Lot_Size,Garage_Size,Neighborhood_Quality,House_Price,Building_Age,Total_Rooms
0,1360,2,1,0.599637,0,5,2.623829e+05,45,3
1,4272,3,3,4.753014,1,6,9.852609e+05,10,6
2,3592,1,2,3.634823,0,9,7.779774e+05,10,3
3,966,1,2,2.730667,1,8,2.296989e+05,49,3
4,4926,2,1,4.699073,0,8,1.041741e+06,33,3


In [33]:
X=df.drop('House_Price' , axis=1)
y=df['House_Price']

print(X.columns.tolist())

['Square_Footage', 'Num_Bedrooms', 'Num_Bathrooms', 'Lot_Size', 'Garage_Size', 'Neighborhood_Quality', 'Building_Age', 'Total_Rooms']


In [34]:
# Train-Test Split
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

Train: (800, 8), Test: (200, 8)


In [35]:
# Scale
scaler=StandardScaler()
X_train=scaler.fit_transform(X_train)
X_test=scaler.transform(X_test)

In [36]:
results=[]

def model_results(name,y_test,y_pred):
    mae  = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2   = r2_score(y_test, y_pred)
    results.append({"Model": name, "MAE": round(mae, 2), 
                    "RMSE": round(rmse, 2), "R2": round(r2, 4)})
    print(f"{name} → MAE: {mae:.2f} | RMSE: {rmse:.2f} | R2: {r2:.4f}")
    

In [37]:
# Linear Regression

regression=LinearRegression()
regression.fit(X_train,y_train)
y_pred=regression.predict(X_test)
model_results("Linear Regression" , y_test , y_pred)

Linear Regression → MAE: 8174.58 | RMSE: 10071.48 | R2: 0.9984


In [38]:
# Polynomial Regression

poly_model=Pipeline([
    ("poly" , PolynomialFeatures(degree=2, include_bias=False)),
    ("regression",   LinearRegression())
]
)
poly_model.fit(X_train,y_train)
y_pred=poly_model.predict(X_test)
model_results("Polynomial Regression" , y_test , y_pred)

Polynomial Regression → MAE: 8311.02 | RMSE: 10187.32 | R2: 0.9984


In [39]:
# Ridge

for alpha in [0.01, 0.1, 1, 10, 100]:
    ridge = Ridge(alpha=alpha)
    ridge.fit(X_train, y_train)
    y_pred=ridge.predict(X_test)
    model_results(f"Ridge α={alpha}", y_test, y_pred)

Ridge α=0.01 → MAE: 8175.22 | RMSE: 10071.96 | R2: 0.9984
Ridge α=0.1 → MAE: 8180.98 | RMSE: 10076.24 | R2: 0.9984
Ridge α=1 → MAE: 8241.78 | RMSE: 10123.51 | R2: 0.9984
Ridge α=10 → MAE: 9045.58 | RMSE: 10996.33 | R2: 0.9981
Ridge α=100 → MAE: 26992.33 | RMSE: 31583.62 | R2: 0.9845


In [40]:
# Lasso

lasso = Lasso(alpha=1.0)
lasso.fit(X_train, y_train)
y_pred = lasso.predict(X_test)
model_results("Lasso α=1.0", y_test, y_pred)

# Katsayı yorumu
coef_df = pd.DataFrame({
    "Feature": X.columns,
    "Coefficient": lasso.coef_
}).sort_values("Coefficient", ascending=False)
print(coef_df)

Lasso α=1.0 → MAE: 8174.77 | RMSE: 10071.59 | R2: 0.9984
                Feature    Coefficient
0        Square_Footage  249787.274245
3              Lot_Size   19087.187044
1          Num_Bedrooms   14960.168534
2         Num_Bathrooms    6945.687889
4           Garage_Size    4218.575667
5  Neighborhood_Quality     334.284983
7           Total_Rooms    -510.720666
6          Building_Age  -20660.992512


In [41]:
# ElasticNet

for alpha in [0.01, 0.1, 1]:
    elastic = ElasticNet(alpha=alpha, l1_ratio=0.5)
    elastic.fit(X_train, y_train)
    y_pred = elastic.predict(X_test)
    model_results(f"ElasticNet α={alpha}", y_test, y_pred)

ElasticNet α=0.01 → MAE: 8464.94 | RMSE: 10337.01 | R2: 0.9983
ElasticNet α=0.1 → MAE: 14133.06 | RMSE: 16957.68 | R2: 0.9955
ElasticNet α=1 → MAE: 75065.05 | RMSE: 86814.09 | R2: 0.8831


In [42]:
results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))

                Model      MAE     RMSE     R2
    Linear Regression  8174.58 10071.48 0.9984
Polynomial Regression  8311.02 10187.32 0.9984
         Ridge α=0.01  8175.22 10071.96 0.9984
          Ridge α=0.1  8180.98 10076.24 0.9984
            Ridge α=1  8241.78 10123.51 0.9984
           Ridge α=10  9045.58 10996.33 0.9981
          Ridge α=100 26992.33 31583.62 0.9845
          Lasso α=1.0  8174.77 10071.59 0.9984
    ElasticNet α=0.01  8464.94 10337.01 0.9983
     ElasticNet α=0.1 14133.06 16957.68 0.9955
       ElasticNet α=1 75065.05 86814.09 0.8831


In [45]:
# HOUSE PRICE PREDICTION 

square_footage       = float(input("Square Footage: "))
num_bedrooms         = int(input("Num Bedrooms: "))
num_bathrooms        = int(input("Num Bathrooms: "))
lot_size             = float(input("Lot Size: "))
garage_size          = int(input("Garage Size (0/1/2): "))
neighborhood_quality = int(input("Neighborhood Quality (1-10): "))
building_age         = int(input("Building Age: "))
total_rooms          = num_bedrooms + num_bathrooms

input_data = pd.DataFrame([[
    square_footage, num_bedrooms, num_bathrooms,
    lot_size, garage_size, neighborhood_quality,
    building_age, total_rooms
]], columns=X.columns)

input_scaled = scaler.transform(input_data)
tahmin = regression.predict(input_scaled)

print("\n--- INPUT ---")
print(f"Square Footage:       {square_footage}")
print(f"Num Bedrooms:         {num_bedrooms}")
print(f"Num Bathrooms:        {num_bathrooms}")
print(f"Lot Size:             {lot_size}")
print(f"Garage Size:          {garage_size}")
print(f"Neighborhood Quality: {neighborhood_quality}")
print(f"Building Age:         {building_age}")
print(f"Total Rooms:          {total_rooms}")
print(f"\nEstimated House Price: ${tahmin[0]:,.2f}")


--- INPUT ---
Square Footage:       10000.0
Num Bedrooms:         2
Num Bathrooms:        3
Lot Size:             4.888999
Garage Size:          1
Neighborhood Quality: 7
Building Age:         25
Total Rooms:          5

Estimated House Price: $2,096,434.88
